In [1]:
# Cell 1: Download and extract dataset (idempotent)
import os, subprocess, sys, shutil

WORK_DIR = "/kaggle/working"
DATA_ZIP = os.path.join(WORK_DIR, "gpr.zip")
DATA_DIR = os.path.join(WORK_DIR, "gpr_data")
MARKER = os.path.join(WORK_DIR, "step_download_done.txt")

if os.path.exists(MARKER):
    print("Download step skipped (already done).")
else:
    # Download
    print("Downloading dataset ...")
    cmd1 = f'wget -O "{DATA_ZIP}" "https://zenodo.org/record/14270869/files/MCG_GPR_dataset.zip?download=1"'
    r = os.system(cmd1)
    if r != 0:
        raise RuntimeError("Download failed. Check network or URL.")
    # Unzip
    print("Unzipping...")
    cmd2 = f'unzip -q "{DATA_ZIP}" -d "{DATA_DIR}"'
    r = os.system(cmd2)
    if r != 0:
        raise RuntimeError("Unzip failed.")
    # marker
    with open(MARKER, "w") as f:
        f.write("done\n")
    print("Download and extraction complete.")


--2025-12-03 12:55:23--  https://zenodo.org/record/14270869/files/MCG_GPR_dataset.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.185.48.75, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/14270869/files/MCG_GPR_dataset.zip [following]
--2025-12-03 12:55:25--  https://zenodo.org/records/14270869/files/MCG_GPR_dataset.zip
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 1385387968 (1.3G) [application/octet-stream]
Saving to: ‘/kaggle/working/gpr.zip’

     0K .......... .......... .......... .......... ..........  0% 91.3K 4h6m
    50K .......... .......... .......... .......... ..........  0%  189K 3h3m
   100K .......... .......... .......... .......... ..........  0%  200M 2h2m
   150K .......... .......... .......... .......... ..........  0%  188K 2h1m
   200K .......... .......... .

Unzipping...
Download and extraction complete.


In [2]:
# Cell 2+: Robust preprocessing

import os
import glob
import json
import math
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

import numpy as np
import cv2

# == CONFIGURATION (change these if desired) ==
WORK_DIR = "/kaggle/working"
DATA_DIR = os.path.join(WORK_DIR, "gpr_data")           # input extracted data (from Cell 1)
OUT_DIR = os.path.join(WORK_DIR, "processed")           # outputs saved here
OUT_IMG_DIR = os.path.join(OUT_DIR, "images")
OUT_MASK_DIR = os.path.join(OUT_DIR, "masks_raw")
OUT_MASK_CLEAN_DIR = os.path.join(OUT_DIR, "masks_clean")
OUT_THUMB_DIR = os.path.join(OUT_DIR, "thumbnails")
META_DIR = os.path.join(OUT_DIR, "meta")
MARKER = os.path.join(WORK_DIR, "step_preprocess_done.txt")
PARAMS_JSON = os.path.join(OUT_DIR, "preprocess_params.json")
DATASET_STATS = os.path.join(OUT_DIR, "dataset_stats.json")

# Processing parameters (saved to JSON)
params = {
    "bg_subtraction": "row_median",   # options: row_median, temporal_median (not implemented on single scan), none
    "clahe_clip_limit": 2.0,
    "clahe_tile_grid": [8, 8],
    "gaussian_blur_ksize": 5,         # odd integer
    "adaptive_thresh_blocksize": 11,  # odd integer
    "adaptive_thresh_C": 2,
    "morph_kernel_scale": 0.02,       # fraction of max(image height, width) used to pick kernel
    "morph_open_iters": 1,
    "morph_close_iters": 1,
    "thumbnail_max_dim": 256,
    "max_workers": 8                   
}

# Create folders
for d in (OUT_IMG_DIR, OUT_MASK_DIR, OUT_MASK_CLEAN_DIR, OUT_THUMB_DIR, META_DIR):
    os.makedirs(d, exist_ok=True)

# Idempotent check: skip entire cell if marker exists
if os.path.exists(MARKER):
    print("Preprocessing marker found. Skipping Cell 2+ (already done).")
else:
    # discover input files (common image extensions; fallback to .npy)
    exts = ("**/*.png", "**/*.jpg", "**/*.jpeg", "**/*.bmp", "**/*.tif", "**/*.tiff")
    files = []
    for e in exts:
        files.extend(sorted(Path(DATA_DIR).glob(e)))
    if len(files) == 0:
        files = sorted(Path(DATA_DIR).glob("**/*.npy"))
        using_npy = True if len(files) > 0 else False
    else:
        using_npy = False
    files = [str(p) for p in files]

    if len(files) == 0:
        raise FileNotFoundError(f"No input files found in {DATA_DIR}. Check Cell 1 extraction or adjust DATA_DIR.")

    print(f"Found {len(files)} input files (using_npy={using_npy}). Starting preprocessing with {params['max_workers']} workers...")

    # Helper functions
    def load_image(path):
        if path.lower().endswith(".npy"):
            arr = np.load(path)
            if arr.ndim == 3:  # if there's a channel dim, collapse
                arr = arr[..., 0] if arr.shape[-1] == 1 else arr.mean(axis=-1)
            return arr.astype(np.float32)
        else:
            im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
            if im is None:
                raise RuntimeError(f"Failed to load image: {path}")
            # convert to grayscale if needed
            if im.ndim == 3:
                im = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
            return im.astype(np.float32)

    def save_png(path, arr):
        # expects uint8 or similar
        cv2.imwrite(path, arr)

    def compute_morph_kernel(img_shape):
        # Kernel size scales with image size so operations adapt to large vs small images
        h, w = img_shape[:2]
        size = max(h, w)
        # kernel diameter based on scale fraction
        k = max(3, int(round(params["morph_kernel_scale"] * size)))
        if k % 2 == 0:
            k += 1
        return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))

    def make_thumbnail(img, max_dim=params["thumbnail_max_dim"]):
        h, w = img.shape[:2]
        scale = max_dim / max(h, w)
        nh, nw = int(h * scale), int(w * scale)
        thumb = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
        return thumb

    def preprocess_single(src_path):
        # returns metadata dict
        name = Path(src_path).stem
        out_img_path = os.path.join(OUT_IMG_DIR, name + ".png")
        out_mask_path = os.path.join(OUT_MASK_DIR, name + ".png")
        out_clean_mask_path = os.path.join(OUT_MASK_CLEAN_DIR, name + ".png")
        out_thumb_path = os.path.join(OUT_THUMB_DIR, name + ".png")
        meta_path = os.path.join(META_DIR, name + ".json")

        # skip if outputs exist (idempotent per-file)
        if os.path.exists(out_img_path) and os.path.exists(out_mask_path) and os.path.exists(out_clean_mask_path) and os.path.exists(meta_path):
            # load minimal meta to return
            try:
                with open(meta_path, "r") as f:
                    meta = json.load(f)
                meta["skipped"] = True
                return meta
            except Exception:
                # fall back to reprocessing if meta is corrupt
                pass

        # Load
        try:
            arr = load_image(src_path)
        except Exception as e:
            return {"name": name, "error": str(e), "src": src_path}

        # Normalize to 0-255 uint8
        arr_min, arr_max = float(np.nanmin(arr)), float(np.nanmax(arr))
        if arr_max - arr_min < 1e-6:
            img_u8 = np.zeros_like(arr, dtype=np.uint8)
        else:
            img_u8 = ((arr - arr_min) / (arr_max - arr_min) * 255.0).astype(np.uint8)

        # Background subtraction (row-wise median)
        if params["bg_subtraction"] == "row_median":
            bg = np.median(img_u8, axis=1, keepdims=True)
            img_bs = (img_u8.astype(np.int16) - bg.astype(np.int16)).astype(np.int16)
            # clip to uint8 range
            img_bs = np.clip(img_bs, 0, 255).astype(np.uint8)
        else:
            img_bs = img_u8

        # Contrast Limited Adaptive Histogram Equalization
        clahe = cv2.createCLAHE(clipLimit=params["clahe_clip_limit"],
                                tileGridSize=tuple(params["clahe_tile_grid"]))
        img_clahe = clahe.apply(img_bs)

        # Smoothing
        k = params["gaussian_blur_ksize"]
        if k % 2 == 0:
            k += 1
        img_smooth = cv2.GaussianBlur(img_clahe, (k, k), 0)

        # Adaptive threshold to produce a raw mask
        blocksize = params["adaptive_thresh_blocksize"]
        if blocksize % 2 == 0:
            blocksize += 1
        raw_mask = cv2.adaptiveThreshold(img_smooth, 255,
                                         cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                         cv2.THRESH_BINARY,
                                         blocksize,
                                         params["adaptive_thresh_C"])

        # Morphological cleaning with adaptive kernel size
        kernel = compute_morph_kernel(img_smooth.shape)
        clean = cv2.morphologyEx(raw_mask, cv2.MORPH_OPEN, kernel, iterations=params["morph_open_iters"])
        clean = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, kernel, iterations=params["morph_close_iters"])

        # Optional contour filtering: remove tiny components based on area fraction
        contours, _ = cv2.findContours(clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        h, w = clean.shape[:2]
        area_thresh = max(20, int(0.0005 * h * w))  # 0.05% of image area min
        filtered = np.zeros_like(clean)
        kept = 0
        for c in contours:
            a = cv2.contourArea(c)
            if a >= area_thresh:
                cv2.drawContours(filtered, [c], -1, 255, -1)
                kept += 1
        clean_mask = filtered if kept > 0 else clean

        # Save outputs
        try:
            save_png(out_img_path, img_smooth)
            save_png(out_mask_path, raw_mask)
            save_png(out_clean_mask_path, clean_mask)
            thumb = make_thumbnail(img_smooth)
            save_png(out_thumb_path, thumb)
        except Exception as e:
            return {"name": name, "error": f"save_failed: {e}", "src": src_path}

        # Compute simple image-level stats for dataset analysis
        fg_fraction = float(np.count_nonzero(clean_mask) / (h * w))
        mean_intensity = float(np.mean(img_smooth))
        std_intensity = float(np.std(img_smooth))

        meta = {
            "name": name,
            "src": src_path,
            "out_img": out_img_path,
            "out_mask": out_mask_path,
            "out_clean_mask": out_clean_mask_path,
            "out_thumb": out_thumb_path,
            "meta_path": meta_path,
            "shape": [int(h), int(w)],
            "area_thresh": int(area_thresh),
            "foreground_fraction": fg_fraction,
            "mean_intensity": mean_intensity,
            "std_intensity": std_intensity,
            "skipped": False
        }

        # write per-image metadata
        try:
            with open(meta_path, "w") as f:
                json.dump(meta, f, indent=2)
        except Exception as e:
            meta["meta_write_error"] = str(e)

        return meta

    # Parallel execution with progress bar
    results = []
    with ThreadPoolExecutor(max_workers=params["max_workers"]) as ex:
        futures = {ex.submit(preprocess_single, p): p for p in files}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Processing images"):
            try:
                res = fut.result()
            except Exception as e:
                res = {"error": f"exception: {e}"}
            results.append(res)

    # Save dataset-wide stats
    valid = [r for r in results if r and ("error" not in r)]
    stats = {
        "num_inputs": len(files),
        "num_processed": len(valid),
        "params": params,
        "example_meta": valid[0] if len(valid)>0 else {},
    }
    # compute aggregated stats (mean foreground fraction etc.)
    fg_fracs = [r.get("foreground_fraction", 0.0) for r in valid]
    if len(fg_fracs) > 0:
        stats["mean_foreground_fraction"] = float(np.mean(fg_fracs))
        stats["median_foreground_fraction"] = float(np.median(fg_fracs))
        stats["std_foreground_fraction"] = float(np.std(fg_fracs))
    else:
        stats["mean_foreground_fraction"] = 0.0

    with open(DATASET_STATS, "w") as f:
        json.dump(stats, f, indent=2)

    # write params JSON for reproducibility
    with open(PARAMS_JSON, "w") as f:
        json.dump(params, f, indent=2)

    # write marker to indicate completion
    with open(MARKER, "w") as f:
        f.write("done\n")

    print("Preprocessing finished. Processed:", stats["num_processed"], "of", stats["num_inputs"])
    print("Dataset stats:", DATASET_STATS)
    print("Per-image metadata and outputs saved under:", OUT_DIR)


Found 9066 input files (using_npy=False). Starting preprocessing with 8 workers...


Processing images:   0%|          | 0/9066 [00:00<?, ?it/s]

Preprocessing finished. Processed: 9066 of 9066
Dataset stats: /kaggle/working/processed/dataset_stats.json
Per-image metadata and outputs saved under: /kaggle/working/processed


In [4]:
# Cell 3: Hybrid dataset builder (You Only Look Once detection + U-Net segmentation)
# Uses: /kaggle/working/processed/images and /kaggle/working/processed/masks_clean
# Creates: /kaggle/working/dataset_hybrid with images, masks, detection labels, polygons, yaml files

import os, glob, shutil, json, random, math
from pathlib import Path
from tqdm.auto import tqdm
import cv2
import numpy as np

# ---------------- CONFIG ----------------
WORK_DIR = "/kaggle/working"
PROCESSED_DIR = os.path.join(WORK_DIR, "processed")

IMG_DIR = os.path.join(PROCESSED_DIR, "images")
CLEAN_MASK_DIR = os.path.join(PROCESSED_DIR, "masks_clean")

OUT_DIR = os.path.join(WORK_DIR, "dataset_hybrid")
MARKER = os.path.join(WORK_DIR, "step_dataset_done_hybrid.txt")

TRAIN_FRAC = 0.75
VAL_FRAC = 0.15
TEST_FRAC = 0.10
RANDOM_SEED = 42

CLASS_NAMES = ["anomaly"]

paths = {
    "images_train": os.path.join(OUT_DIR, "images", "train"),
    "images_val": os.path.join(OUT_DIR, "images", "val"),
    "images_test": os.path.join(OUT_DIR, "images", "test"),
    "masks_train": os.path.join(OUT_DIR, "masks", "train"),
    "masks_val": os.path.join(OUT_DIR, "masks", "val"),
    "masks_test": os.path.join(OUT_DIR, "masks", "test"),
    "labels_det_train": os.path.join(OUT_DIR, "labels_detection", "train"),
    "labels_det_val": os.path.join(OUT_DIR, "labels_detection", "val"),
    "labels_det_test": os.path.join(OUT_DIR, "labels_detection", "test"),
    "polygons": os.path.join(OUT_DIR, "polygons_json")
}

# ---------------- IDEMPOTENT CHECK ----------------
if os.path.exists(MARKER):
    print("Hybrid dataset already created — skipping Cell 3.")
else:
    # Make sure input folders exist
    if not os.path.exists(IMG_DIR) or not os.path.exists(CLEAN_MASK_DIR):
        raise FileNotFoundError("Processed images or masks_clean not found. Check Cell 2 output paths.")

    # Create output folders
    for p in paths.values():
        os.makedirs(p, exist_ok=True)

    # Collect processed images and masks
    img_files = sorted(glob.glob(os.path.join(IMG_DIR, "*.png")))
    mask_files = sorted(glob.glob(os.path.join(CLEAN_MASK_DIR, "*.png")))

    img_map = {Path(p).stem: p for p in img_files}
    mask_map = {Path(p).stem: p for p in mask_files}
    common_keys = sorted([k for k in img_map if k in mask_map])

    if len(common_keys) == 0:
        raise FileNotFoundError("No matching processed images and clean masks found. Check Cell 2 outputs.")

    # Split into train / validation / test
    random.seed(RANDOM_SEED)
    keys = common_keys.copy()
    random.shuffle(keys)
    n = len(keys)
    n_test = max(1, int(math.floor(TEST_FRAC * n)))
    n_val = max(1, int(math.floor(VAL_FRAC * n)))
    n_train = n - n_val - n_test

    train_keys = keys[:n_train]
    val_keys = keys[n_train:n_train + n_val]
    test_keys = keys[n_train + n_val:]

    splits = [("train", train_keys), ("val", val_keys), ("test", test_keys)]
    stats = {"total": n, "train": len(train_keys), "val": len(val_keys), "test": len(test_keys)}
    print("Dataset split:", stats)

    # Helper: write You Only Look Once detection labels (single class, normalized)
    def write_yolo_det_label(label_path, boxes, img_w, img_h):
        lines = []
        for (xmin, ymin, xmax, ymax) in boxes:
            xc = (xmin + xmax) / 2.0
            yc = (ymin + ymax) / 2.0
            bw = xmax - xmin
            bh = ymax - ymin
            xc_n = xc / img_w
            yc_n = yc / img_h
            bw_n = bw / img_w
            bh_n = bh / img_h
            lines.append(f"0 {xc_n:.6f} {yc_n:.6f} {bw_n:.6f} {bh_n:.6f}")
        with open(label_path, "w") as f:
            f.write("\n".join(lines))

    # Main loop: copy files and generate labels + polygons
    for split_name, keys_list in tqdm(splits, desc="Splits"):
        for k in tqdm(keys_list, desc=f"Processing {split_name}", leave=False):
            img_src = img_map[k]
            mask_src = mask_map[k]

            img_dst = os.path.join(paths[f"images_{split_name}"], Path(img_src).name)
            mask_dst = os.path.join(paths[f"masks_{split_name}"], Path(mask_src).name)
            label_dst = os.path.join(paths[f"labels_det_{split_name}"], Path(img_src).stem + ".txt")

            # Copy image and mask if not already present
            if not os.path.exists(img_dst):
                shutil.copy(img_src, img_dst)
            if not os.path.exists(mask_dst):
                shutil.copy(mask_src, mask_dst)

            # Load mask for contour extraction
            mask = cv2.imread(mask_src, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                write_yolo_det_label(label_dst, [], 1, 1)
                poly_data = {"name": k, "polygons": []}
            else:
                _, bw = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
                contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                boxes = []
                polygons = []
                h, w = bw.shape[:2]
                for cnt in contours:
                    area = cv2.contourArea(cnt)
                    if area < 5:
                        continue
                    x, y, ww, hh = cv2.boundingRect(cnt)
                    boxes.append((x, y, x + ww, y + hh))

                    epsilon = 0.01 * cv2.arcLength(cnt, True)
                    approx = cv2.approxPolyDP(cnt, epsilon, True)
                    poly_points = [(int(p[0][0]), int(p[0][1])) for p in approx]
                    polygons.append(poly_points)

                write_yolo_det_label(label_dst, boxes, w, h)
                poly_data = {"name": k, "polygons": polygons}

            poly_json_path = os.path.join(paths["polygons"], Path(img_src).stem + ".json")
            with open(poly_json_path, "w") as pf:
                json.dump(poly_data, pf)

    # Data yaml for segmentation (images + masks)
    data_seg_yaml = os.path.join(OUT_DIR, "data_seg.yaml")
    with open(data_seg_yaml, "w") as f:
        f.write(f"train: {os.path.abspath(paths['images_train'])}\n")
        f.write(f"val: {os.path.abspath(paths['images_val'])}\n")
        f.write("nc: 1\n")
        f.write(f"names: {CLASS_NAMES}\n")

    # Data yaml for detection (You Only Look Once)
    data_det_yaml = os.path.join(OUT_DIR, "data_det.yaml")
    with open(data_det_yaml, "w") as f:
        f.write(f"train: {os.path.abspath(paths['images_train'])}\n")
        f.write(f"val: {os.path.abspath(paths['images_val'])}\n")
        f.write("nc: 1\n")
        f.write(f"names: {CLASS_NAMES}\n")

    manifest = {
        "splits": stats,
        "paths": paths,
        "data_seg_yaml": data_seg_yaml,
        "data_det_yaml": data_det_yaml,
        "note": "Images and masks are ready for segmentation. Detection labels are ready for You Only Look Once. Polygons are stored separately."
    }
    with open(os.path.join(OUT_DIR, "manifest.json"), "w") as mf:
        json.dump(manifest, mf, indent=2)

    with open(MARKER, "w") as f:
        f.write("done\n")

    print("Hybrid dataset creation complete.")
    print("Manifest:", os.path.join(OUT_DIR, "manifest.json"))
    print("Segmentation yaml:", data_seg_yaml)
    print("Detection yaml:", data_det_yaml)


Hybrid dataset already created — skipping Cell 3.


In [6]:
# Cell 4: Train YOLO

import os

# Install ultralytics if not already installed
try:
    import ultralytics
except ImportError:
    !pip install ultralytics -q
    import ultralytics

from ultralytics import YOLO

DATA_DET_YAML = "/kaggle/working/dataset_hybrid/data_det.yaml"
MODELS_DIR = "/kaggle/working/models"
os.makedirs(MODELS_DIR, exist_ok=True)

YOLO_WEIGHTS_PATH = os.path.join(MODELS_DIR, "yolo_gpr_best.pt")

if os.path.exists(YOLO_WEIGHTS_PATH):
    print("You Only Look Once model already trained and saved at:")
    print(YOLO_WEIGHTS_PATH)
else:
    print("Training You Only Look Once detection model...")
    model = YOLO("yolov8n.pt")  # nano model for speed

    results = model.train(
        data=DATA_DET_YAML,
        epochs=30,
        imgsz=640,
        batch=8,
        project="/kaggle/working/runs_yolo_det",
        name="gpr_det",
        exist_ok=True
    )

    # Find best.pt and copy to fixed path
    import glob, shutil
    candidates = glob.glob("/kaggle/working/runs_yolo_det/**/weights/best.pt", recursive=True)
    if not candidates:
        raise FileNotFoundError("Could not locate best.pt after training.")
    best_path = sorted(candidates)[-1]

    print("Copying best weights to:", YOLO_WEIGHTS_PATH)
    shutil.copy(best_path, YOLO_WEIGHTS_PATH)
    print("Training done. You Only Look Once model saved.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 109.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 53.1 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:832: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30      1.21G          0      2.119          0          0        640: 100% ━━━━━━━━━━━━ 631/631 10.2it/s 1:02<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 64/64 29.2it/s 2.2s0.1s
                   all       1009          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, can not compute metrics without labels


/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:832: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      1.22G          0    0.01389          0          0        640: 100% ━━━━━━━━━━━━ 631/631 10.3it/s 1:010.1sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 64/64 29.1it/s 2.2s0.1s
                   all       1009          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, can not compute metrics without labels


/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:832: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide


... (53 outputs left out)

/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:832: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide



30 epochs completed in 0.526 hours.
Optimizer stripped from /kaggle/working/runs_yolo_det/gpr_det/weights/last.pt, 6.2MB
Optimizer stripped from /kaggle/working/runs_yolo_det/gpr_det/weights/best.pt, 6.2MB

Validating /kaggle/working/runs_yolo_det/gpr_det/weights/best.pt...
Ultralytics 8.3.234 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 64/64 28.2it/s 2.3s0.1s


/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:653: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:697: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:697: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/metrics.py:697: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0),

                   all       1009          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, can not compute metrics without labels
Speed: 0.1ms preprocess, 1.4ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /kaggle/working/runs_yolo_det/gpr_det
Copying best weights to: /kaggle/working/models/yolo_gpr_best.pt
Training done. You Only Look Once model saved.


In [7]:
# Cell 5: Train U-Net segmentation model 

import os, glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
from tqdm.auto import tqdm

DATASET_DIR = "/kaggle/working/dataset_hybrid"
IMG_TRAIN_DIR = os.path.join(DATASET_DIR, "images", "train")
MASK_TRAIN_DIR = os.path.join(DATASET_DIR, "masks", "train")
IMG_VAL_DIR = os.path.join(DATASET_DIR, "images", "val")
MASK_VAL_DIR = os.path.join(DATASET_DIR, "masks", "val")

MODELS_DIR = "/kaggle/working/models"
os.makedirs(MODELS_DIR, exist_ok=True)
UNET_WEIGHTS_PATH = os.path.join(MODELS_DIR, "unet_gpr_best.pth")

if os.path.exists(UNET_WEIGHTS_PATH):
    print("U-Net model already trained and saved at:")
    print(UNET_WEIGHTS_PATH)
else:
    # ----- U-Net definition -----
    class DoubleConv(nn.Module):
        def __init__(self, in_ch, out_ch):
            super().__init__()
            self.conv = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            )

        def forward(self, x):
            return self.conv(x)

    class UNetSmall(nn.Module):
        def __init__(self, n_classes=1):
            super().__init__()
            self.down1 = DoubleConv(1, 32)
            self.pool1 = nn.MaxPool2d(2)
            self.down2 = DoubleConv(32, 64)
            self.pool2 = nn.MaxPool2d(2)
            self.down3 = DoubleConv(64, 128)
            self.pool3 = nn.MaxPool2d(2)

            self.bottleneck = DoubleConv(128, 256)

            self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
            self.conv3 = DoubleConv(256, 128)
            self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
            self.conv2 = DoubleConv(128, 64)
            self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
            self.conv1 = DoubleConv(64, 32)

            self.out_conv = nn.Conv2d(32, n_classes, 1)

        def forward(self, x):
            c1 = self.down1(x)
            p1 = self.pool1(c1)
            c2 = self.down2(p1)
            p2 = self.pool2(c2)
            c3 = self.down3(p2)
            p3 = self.pool3(c3)

            bn = self.bottleneck(p3)

            u3 = self.up3(bn)
            u3 = torch.cat([u3, c3], dim=1)
            c3 = self.conv3(u3)
            u2 = self.up2(c3)
            u2 = torch.cat([u2, c2], dim=1)
            c2 = self.conv2(u2)
            u1 = self.up1(c2)
            u1 = torch.cat([u1, c1], dim=1)
            c1 = self.conv1(u1)

            out = self.out_conv(c1)
            return out

    class GPRSegDataset(Dataset):
        def __init__(self, img_dir, mask_dir, img_size=256):
            self.img_paths = sorted(glob.glob(os.path.join(img_dir, "*.png")))
            self.mask_paths = [os.path.join(mask_dir, os.path.basename(p)) for p in self.img_paths]
            self.img_size = img_size

        def __len__(self):
            return len(self.img_paths)

        def __getitem__(self, idx):
            img = cv2.imread(self.img_paths[idx], cv2.IMREAD_GRAYSCALE)
            mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (self.img_size, self.img_size))
            mask = cv2.resize(mask, (self.img_size, self.img_size))
            img = img.astype(np.float32) / 255.0
            mask = (mask > 127).astype(np.float32)
            img = np.expand_dims(img, 0)
            mask = np.expand_dims(mask, 0)
            return torch.from_numpy(img), torch.from_numpy(mask)

    train_ds = GPRSegDataset(IMG_TRAIN_DIR, MASK_TRAIN_DIR, img_size=256)
    val_ds = GPRSegDataset(IMG_VAL_DIR, MASK_VAL_DIR, img_size=256)

    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = UNetSmall(n_classes=1).to(device)

    bce = nn.BCEWithLogitsLoss()

    def dice_loss(logits, targets, eps=1e-6):
        probs = torch.sigmoid(logits)
        num = 2 * (probs * targets).sum(dim=(2,3))
        den = probs.sum(dim=(2,3)) + targets.sum(dim=(2,3)) + eps
        dice = 1 - num / den
        return dice.mean()

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val_loss = float("inf")
    EPOCHS = 20

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch} [train]", leave=False):
            imgs = imgs.to(device)
            masks = masks.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss_bce = bce(logits, masks)
            loss_dice = dice_loss(logits, masks)
            loss = loss_bce + loss_dice
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * imgs.size(0)
        avg_train_loss = total_loss / len(train_ds)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch} [val]", leave=False):
                imgs = imgs.to(device)
                masks = masks.to(device)
                logits = model(imgs)
                loss_bce = bce(logits, masks)
                loss_dice = dice_loss(logits, masks)
                loss = loss_bce + loss_dice
                val_loss += loss.item() * imgs.size(0)
        avg_val_loss = val_loss / len(val_ds)
        print(f"Epoch {epoch}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), UNET_WEIGHTS_PATH)
            print("Saved best model so far to:", UNET_WEIGHTS_PATH)

    print("Training finished. Best U-Net model stored at:", UNET_WEIGHTS_PATH)


Epoch 1 [train]:   0%|          | 0/631 [00:00<?, ?it/s]

Epoch 1 [val]:   0%|          | 0/127 [00:00<?, ?it/s]

Epoch 1: train_loss=0.4995, val_loss=0.3776
Saved best model so far to: /kaggle/working/models/unet_gpr_best.pth


Epoch 2 [train]:   0%|          | 0/631 [00:00<?, ?it/s]

Epoch 2 [val]:   0%|          | 0/127 [00:00<?, ?it/s]

Epoch 2: train_loss=0.3579, val_loss=0.3706
Saved best model so far to: /kaggle/working/models/unet_gpr_best.pth


... (50 outputs left out)

Epoch 19: train_loss=0.2371, val_loss=0.2360


Epoch 20 [train]:   0%|          | 0/631 [00:00<?, ?it/s]

Epoch 20 [val]:   0%|          | 0/127 [00:00<?, ?it/s]

Epoch 20: train_loss=0.2359, val_loss=0.2123
Saved best model so far to: /kaggle/working/models/unet_gpr_best.pth
Training finished. Best U-Net model stored at: /kaggle/working/models/unet_gpr_best.pth


In [8]:
# Cell 6: Export trained models for inference
# - Export You Only Look Once best model to ONNX
# - Export U-Net best model to TorchScript

import os
import torch
import torch.nn as nn

from ultralytics import YOLO

MODELS_DIR = "/kaggle/working/models"
os.makedirs(MODELS_DIR, exist_ok=True)

YOLO_WEIGHTS_PATH = os.path.join(MODELS_DIR, "yolo_gpr_best.pt")
YOLO_ONNX_PATH = os.path.join(MODELS_DIR, "yolo_gpr_best.onnx")

UNET_WEIGHTS_PATH = os.path.join(MODELS_DIR, "unet_gpr_best.pth")
UNET_TORCHSCRIPT_PATH = os.path.join(MODELS_DIR, "unet_gpr_best_ts.pt")

# ----- Export You Only Look Once to ONNX -----
if not os.path.exists(YOLO_WEIGHTS_PATH):
    print("You Only Look Once weights not found, run Cell 4 first.")
else:
    if os.path.exists(YOLO_ONNX_PATH):
        print("You Only Look Once ONNX model already exists at:", YOLO_ONNX_PATH)
    else:
        print("Exporting You Only Look Once model to ONNX...")
        yolo_model = YOLO(YOLO_WEIGHTS_PATH)
        yolo_model.export(format="onnx", opset=12, imgsz=640, half=False)
        # Ultralytics saves in the same folder as weights (model_name.onnx)
        # Move or rename if needed
        # Find the produced onnx
        import glob, shutil
        candidates = glob.glob("*.onnx") + glob.glob("runs/**/**.onnx", recursive=True)
        if candidates:
            produced = sorted(candidates)[-1]
            shutil.copy(produced, YOLO_ONNX_PATH)
            print("You Only Look Once ONNX saved to:", YOLO_ONNX_PATH)
        else:
            print("Warning: ONNX file not found after export. Check ultralytics output.")

# ----- Export U-Net to TorchScript -----
if not os.path.exists(UNET_WEIGHTS_PATH):
    print("U-Net weights not found, run Cell 5 first.")
else:
    if os.path.exists(UNET_TORCHSCRIPT_PATH):
        print("U-Net TorchScript model already exists at:", UNET_TORCHSCRIPT_PATH)
    else:
        print("Exporting U-Net model to TorchScript...")

        class DoubleConv(nn.Module):
            def __init__(self, in_ch, out_ch):
                super().__init__()
                self.conv = nn.Sequential(
                    nn.Conv2d(in_ch, out_ch, 3, padding=1),
                    nn.BatchNorm2d(out_ch),
                    nn.ReLU(inplace=True),
                    nn.Conv2d(out_ch, out_ch, 3, padding=1),
                    nn.BatchNorm2d(out_ch),
                    nn.ReLU(inplace=True),
                )

            def forward(self, x):
                return self.conv(x)

        class UNetSmall(nn.Module):
            def __init__(self, n_classes=1):
                super().__init__()
                self.down1 = DoubleConv(1, 32)
                self.pool1 = nn.MaxPool2d(2)
                self.down2 = DoubleConv(32, 64)
                self.pool2 = nn.MaxPool2d(2)
                self.down3 = DoubleConv(64, 128)
                self.pool3 = nn.MaxPool2d(2)

                self.bottleneck = DoubleConv(128, 256)

                self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
                self.conv3 = DoubleConv(256, 128)
                self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
                self.conv2 = DoubleConv(128, 64)
                self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
                self.conv1 = DoubleConv(64, 32)

                self.out_conv = nn.Conv2d(32, n_classes, 1)

            def forward(self, x):
                c1 = self.down1(x)
                p1 = self.pool1(c1)
                c2 = self.down2(p1)
                p2 = self.pool2(c2)
                c3 = self.down3(p2)
                p3 = self.pool3(c3)

                bn = self.bottleneck(p3)

                u3 = self.up3(bn)
                u3 = torch.cat([u3, c3], dim=1)
                c3 = self.conv3(u3)
                u2 = self.up2(c3)
                u2 = torch.cat([u2, c2], dim=1)
                c2 = self.conv2(u2)
                u1 = self.up1(c2)
                u1 = torch.cat([u1, c1], dim=1)
                c1 = self.conv1(u1)

                out = self.out_conv(c1)
                return out

        device = "cpu"
        unet = UNetSmall(n_classes=1).to(device)
        state = torch.load(UNET_WEIGHTS_PATH, map_location=device)
        unet.load_state_dict(state)
        unet.eval()

        dummy_input = torch.randn(1, 1, 256, 256, device=device)
        traced = torch.jit.trace(unet, dummy_input)
        traced.save(UNET_TORCHSCRIPT_PATH)
        print("U-Net TorchScript saved to:", UNET_TORCHSCRIPT_PATH)


Exporting You Only Look Once model to ONNX...
Ultralytics 8.3.234 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/kaggle/working/models/yolo_gpr_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.9 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.11.13 environment at: /usr
Resolved 14 packages in 301ms
Prepared 4 packages in 3.70s
Installed 4 packages in 11ms
 + coloredlogs==15.0.1
 + humanfriendly==10.0
 + onnxruntime-gpu==1.23.2
 + onnxslim==0.1.78

requirements: AutoUpdate success ✅ 4.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1

In [9]:
# Zip everything in /models so you can download and reuse 
!cd /kaggle/working && zip -r gpr_models.zip models

print("Created archive:", "/kaggle/working/gpr_models.zip")


  adding: models/ (stored 0%)
  adding: models/unet_gpr_best.pth (deflated 8%)
  adding: models/unet_gpr_best_ts.pt (deflated 8%)
  adding: models/yolo_gpr_best.pt (deflated 9%)
  adding: models/yolo_gpr_best.onnx (deflated 14%)
Created archive: /kaggle/working/gpr_models.zip
